# Software Development Lifecycle using Lang-Graph

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0.5
)

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.types import Command, interrupt
import datetime

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
config = {"configurables": {"thread_id" : "1", "time" :  f"{datetime.datetime.now()}"}}

In [ ]:
from tools.planner import create, append, rewrite, read_plan, summarize_plan, update_plan

In [ ]:
tools = [create, append, rewrite, read_plan, summarize_plan, update_plan]

In [ ]:
planner = llm.bind_tools(tools = tools)

def planning_agent(state:State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("Planner", planner)
graph_builder.add_node("Tools". ToolNode(tools))

graph_builder.add_edge(START, "Planner")
graph_builder.add_conditional_edges(
    "Planner",
    tools_condition()
)
graph_builder.add_edge("Tools", "Planner")

graph = graph_builder.compile(checkpointer=memory)